# EVolvAI: Physics-Informed Generative Framework

This notebook trains the Physics-Informed TCN-VAE (with LinDistFlow constraints) on Google Colab with GPU acceleration.

In [ ]:
!pip install torch numpy

In [ ]:

import torch
import torch.optim as optim
import os

from generative_core import config
from generative_core.models import GenerativeCounterfactualVAE, vae_loss_function
from generative_core.data_loader import get_dataloader
from generative_core.physics_loss import LinDistFlowLoss

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = GenerativeCounterfactualVAE().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    loader = get_dataloader()

    baseline_cond = torch.tensor(
        config.BASELINE_CONDITION, dtype=torch.float32, device=device,
    )

    physics_engine = LinDistFlowLoss(device)

    model.train()
    
    epochs = config.EPOCHS
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        num_batches = 0

        for batch in loader:
            x = batch.permute(0, 2, 1).to(device)
            cond = baseline_cond.unsqueeze(0).expand(x.size(0), -1)

            optimizer.zero_grad()
            recon, mu, logvar = model(x, cond)

            ev_demand = recon[:, :config.NUM_NODES, :].permute(0, 2, 1)
            
            pen_v, pen_therm, pen_xfmr = physics_engine(ev_demand)
            physics_loss = (config.LAMBDA_VOLT * pen_v +
                            config.LAMBDA_THERMAL * pen_therm +
                            config.LAMBDA_XFMR * pen_xfmr)

            loss = vae_loss_function(recon, x, mu, logvar, physics_loss)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_NORM)

            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1

        avg = epoch_loss / max(num_batches, 1)
        print(f"  Epoch {epoch:>3}/{epochs}  avg_loss={avg:.6f}")

    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
    print(f"Checkpoint saved -> {config.MODEL_SAVE_PATH}")
    
    return model, device

model, device = run_training()


In [ ]:

import numpy as np

model.eval()
for name, spec in config.SCENARIOS.items():
    with torch.no_grad():
        z = torch.randn(1, config.LATENT_DIM, device=device)
        cond = torch.tensor([spec["condition"]], dtype=torch.float32, device=device)
        out = model.decode(z, cond)
        
        # Take the EV demand part and shape it
        demand = out[:, :config.NUM_NODES, :].squeeze(0).permute(1, 0).cpu().numpy()
        print(f"[{name}] {spec['description']}  shape={demand.shape}")
        
        path = os.path.join(config.OUTPUT_DIR, f"{name}.npy")
        np.save(path, demand)
        print(f"  -> saved {path}")
print("All counterfactual scenarios generated.")
